# 02 — Modelo de referencia (línea de base)

**Tesis:** Estratificación de riesgo clínico consciente de la equidad.
**Objetivo de este notebook:** entrenar un modelo de riesgo de **mortalidad** con desempeño comparable al estado del arte, que servirá como *sujeto* de la auditoría de equidad (notebook 03). **No** es la aportación: es la base sobre la que mediremos el sesgo.

Predictores permitidos (sin fuga de información): edad, sexo y comorbilidades disponibles al ingreso.
Se **excluyen** `UCI`, `INTUBADO` y todo lo derivado de `FECHA_DEF`.
Como anticipo del análisis de equidad, se evalúa el **AUROC por entidad federativa**.


## 1. Carga del dataset limpio

In [1]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix)
from sklearn.calibration import calibration_curve

import tesis_common as tc

# cargar_cohorte valida procedencia contra el manifiesto del notebook 01:
# columnas requeridas, conteo de filas y bandera de cohorte. Falla rápido si el
# parquet en disco quedó de una corrida anterior.
df, manifiesto = tc.cargar_cohorte()

print(f"Cohorte: {len(df):,} filas | solo_confirmados={manifiesto['solo_confirmados']} "
      f"| tasa_def={manifiesto['tasa_defuncion']} | generada {manifiesto['generado_en']}")
df.head(3)

Cohorte: 2,526,649 filas | solo_confirmados=True | tasa_def=0.056003 | generada 2026-07-31T00:00:20


,ID_REGISTRO,hospitalizado,defuncion,EDAD,grupo_edad,sexo_mujer,entidad_nombre,MUNICIPIO_RES,CLASIFICACION_FINAL,DIABETES_bin,...,EPOC_na,ASMA_na,INMUSUPR_na,HIPERTENSION_na,OTRA_COM_na,CARDIOVASCULAR_na,OBESIDAD_na,RENAL_CRONICA_na,TABAQUISMO_na,NEUMONIA_na
0,1009d6,0,0,36.0,18-39,1,VERACRUZ,85,3,1.0,...,0,0,0,0,0,0,0,0,0,0
1,0f6295,0,0,63.0,60+,1,SAN LUIS POTOSI,28,3,0.0,...,0,0,0,0,0,0,0,0,0,0
2,gzza9a9,0,0,18.0,18-39,1,CIUDAD DE MEXICO,4,1,0.0,...,0,0,0,0,0,0,0,0,0,0


## 2. Definición de X, y y atributos sensibles

In [2]:
PREDICTORES = tc.PREDICTORES        # EDAD, sexo_mujer y comorbilidades al ingreso
TARGET = tc.TARGET                  # "defuncion"
SENSIBLE = "entidad_nombre"         # eje de equidad; NO se usa como predictor

X = df[PREDICTORES].astype("float32")
y = df[TARGET].astype(int)
grupo = df[SENSIBLE]

print("Predictores:", PREDICTORES)
print("Prevalencia del desenlace:", round(y.mean(), 4),
      "| desbalance ~", int((1 - y.mean()) / y.mean()), ":1")

Predictores: ['EDAD', 'sexo_mujer', 'DIABETES_bin', 'EPOC_bin', 'ASMA_bin', 'INMUSUPR_bin', 'HIPERTENSION_bin', 'OTRA_COM_bin', 'CARDIOVASCULAR_bin', 'OBESIDAD_bin', 'RENAL_CRONICA_bin', 'TABAQUISMO_bin', 'NEUMONIA_bin']
Prevalencia del desenlace: 0.056 | desbalance ~ 16 :1


## 3. Partición entrenamiento / calibración / prueba

Partición en **tres** bloques, definida en `tesis_common.particionar` y por tanto idéntica en los notebooks 03 y 04.

El bloque de **prueba** (25 %, estratificado por el desenlace, semilla 42) se separa primero, de modo que la población auditada es la misma que en corridas anteriores y los resultados siguen siendo comparables. Del bloque restante se talla un 20 % de **calibración**, que nunca se usa para entrenar ni para evaluar. Esto corrige una contaminación de la versión anterior, donde el notebook 04 ajustaba la regresión isotónica sobre la mitad del conjunto de prueba y reportaba métricas sobre la otra mitad.

Se conserva la entidad de cada registro para la evaluación por grupo.


In [3]:
# Partición en tres bloques, definida en tesis_common y por tanto idéntica en
# los notebooks 03 y 04. El bloque de TEST se separa primero (25%, semilla 42),
# de modo que la población auditada es la misma que en corridas anteriores; la
# calibración se talla del bloque de ajuste, nunca de test.
idx_tr, idx_cal, idx_te = tc.particionar(y)

X_tr, y_tr = X.iloc[idx_tr], y.iloc[idx_tr]
X_cal, y_cal = X.iloc[idx_cal], y.iloc[idx_cal]
X_te, y_te = X.iloc[idx_te], y.iloc[idx_te]
g_te = grupo.iloc[idx_te]

# Los tres bloques deben ser disjuntos: si se solaparan, toda métrica de test
# quedaría contaminada y la auditoría de equidad no mediría generalización.
assert not (set(idx_tr) & set(idx_cal)), "train y calib se solapan"
assert not (set(idx_tr) & set(idx_te)), "train y test se solapan"
assert not (set(idx_cal) & set(idx_te)), "calib y test se solapan"

print("Entrenamiento:", X_tr.shape, "| Calibración:", X_cal.shape, "| Prueba:", X_te.shape)
print("Prevalencia  tr/cal/te:", round(y_tr.mean(), 4), round(y_cal.mean(), 4), round(y_te.mean(), 4))

Entrenamiento: (1515988, 13) | Calibración: (378998, 13) | Prueba: (631663, 13)
Prevalencia  tr/cal/te: 0.056 0.056 0.056


## 4. Modelo A — Regresión logística (referencia interpretable)

Con `class_weight='balanced'` para el desbalance, imputación y estandarización.


In [4]:
logit = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=tc.SEMILLA)),
])
logit.fit(X_tr, y_tr)
p_logit = logit.predict_proba(X_te)[:, 1]
print("LOGIT  AUROC:", round(roc_auc_score(y_te, p_logit), 4),
      "| AUPRC:", round(average_precision_score(y_te, p_logit), 4),
      "| Brier:", round(brier_score_loss(y_te, p_logit), 4))

LOGIT  AUROC: 0.9489 | AUPRC: 0.5528 | Brier: 0.0819


## 5. Modelo B — Gradient Boosting calibrado (modelo principal)

`HistGradientBoostingClassifier` maneja valores faltantes de forma nativa y es rápido en millones de filas. Los hiperparámetros viven en `tesis_common.HPARAMS_HGB`, que es ahora la única definición del modelo en toda la cadena.

`class_weight="balanced"` es necesario por el desbalance ~17:1, pero deforma el nivel de las probabilidades: el modelo ordena bien y sobreestima el riesgo absoluto por un factor cercano a 4. Por eso se **calibra con regresión isotónica** sobre el bloque de calibración. Como la transformación isotónica es monótona, el AUROC y el AUPRC no cambian; lo que se corrige es la interpretabilidad clínica de la probabilidad y, con ella, la posibilidad de auditar la calibración por grupo en el notebook 03 sin que el resultado esté dominado por este artefacto global.


In [5]:
hgb = HistGradientBoostingClassifier(**tc.HPARAMS_HGB)
hgb.fit(X_tr, y_tr)
p_te_crudo = hgb.predict_proba(X_te)[:, 1]

# class_weight="balanced" reescala la pérdida: el modelo ORDENA bien pero
# sobreestima el riesgo absoluto ~4x, así que sus probabilidades no son
# interpretables clínicamente ni auditables en calibración. Se corrige con
# regresión isotónica ajustada en el bloque de CALIBRACIÓN (nunca en test).
p_cal_crudo = hgb.predict_proba(X_cal)[:, 1]
calibrador = IsotonicRegression(out_of_bounds="clip").fit(p_cal_crudo, y_cal)
p_hgb = calibrador.predict(p_te_crudo)

# La isotónica es monótona: preserva el orden, luego AUROC y AUPRC se mantienen.
# Lo que corrige es el nivel (Brier, riesgo medio) y la lectura clínica.
print("HGB crudo      AUROC:", round(roc_auc_score(y_te, p_te_crudo), 4),
      "| AUPRC:", round(average_precision_score(y_te, p_te_crudo), 4),
      "| Brier:", round(brier_score_loss(y_te, p_te_crudo), 4),
      "| riesgo medio:", round(float(p_te_crudo.mean()), 4))
print("HGB calibrado  AUROC:", round(roc_auc_score(y_te, p_hgb), 4),
      "| AUPRC:", round(average_precision_score(y_te, p_hgb), 4),
      "| Brier:", round(brier_score_loss(y_te, p_hgb), 4),
      "| riesgo medio:", round(float(p_hgb.mean()), 4))
print("Mortalidad observada en test:", round(float(y_te.mean()), 4))

HGB crudo      AUROC: 0.9504 | AUPRC: 0.5594 | Brier: 0.0866 | riesgo medio: 0.2054
HGB calibrado  AUROC: 0.9503 | AUPRC: 0.5533 | Brier: 0.0323 | riesgo medio: 0.0558
Mortalidad observada en test: 0.056


## 6. Evaluación clínica en el punto de operación

El umbral se fija para alcanzar la **sensibilidad global objetivo de 80 %** (`tesis_common.SENSIBILIDAD_OBJETIVO`), que es el punto de operación propio de una herramienta de triage y **el mismo** que auditan los notebooks 03 y 04.

La versión anterior evaluaba en 0.5, un umbral sin significado en un modelo con `class_weight="balanced"` y distinto del que usaba la auditoría, de modo que las tablas de las tres etapas no eran comparables entre sí.


In [6]:
# Punto de operación clínico compartido (tesis_common.SENSIBILIDAD_OBJETIVO = 0.80).
# Antes se evaluaba en 0.5, un umbral sin significado en un modelo con
# class_weight="balanced" y que además no era el que auditaban los notebooks
# 03 y 04: las tablas de las tres etapas no eran comparables entre sí.
UMBRAL, tpr_obj, fpr_obj = tc.umbral_para_sensibilidad(y_te, p_hgb)
pred = (p_hgb >= UMBRAL).astype(int)

print(f"Umbral (sensibilidad objetivo {tc.SENSIBILIDAD_OBJETIVO:.0%}) = {UMBRAL:.4f}"
      f"  ->  TPR {tpr_obj:.3f} | FPR {fpr_obj:.3f}")
print(classification_report(y_te, pred, digits=3))
print("Matriz de confusión:\n", confusion_matrix(y_te, pred))

Umbral (sensibilidad objetivo 80%) = 0.1562  ->  TPR 0.795 | FPR 0.059
              precision    recall  f1-score   support

           0      0.987     0.941     0.963    596288
           1      0.443     0.795     0.569     35375

    accuracy                          0.933    631663
   macro avg      0.715     0.868     0.766    631663
weighted avg      0.957     0.933     0.941    631663



Matriz de confusión:
 [[560961  35327]
 [  7246  28129]]


## 7. Importancia de variables (permutación)

In [7]:
from sklearn.inspection import permutation_importance
# Muestra para acelerar la permutación
idx = np.random.RandomState(0).choice(len(X_te), size=min(50000, len(X_te)), replace=False)
imp = permutation_importance(hgb, X_te.iloc[idx], y_te.iloc[idx],
                             n_repeats=3, random_state=0, scoring="roc_auc")
imp_df = pd.Series(imp.importances_mean, index=PREDICTORES).sort_values(ascending=False)
print(imp_df.round(4))

NEUMONIA_bin          0.1057
EDAD                  0.0789
sexo_mujer            0.0029
DIABETES_bin          0.0028
RENAL_CRONICA_bin     0.0016
OBESIDAD_bin          0.0011
OTRA_COM_bin          0.0010
HIPERTENSION_bin      0.0009
INMUSUPR_bin          0.0002
TABAQUISMO_bin        0.0001
CARDIOVASCULAR_bin    0.0001
EPOC_bin              0.0000
ASMA_bin              0.0000
dtype: float64


## 8. Anticipo de equidad — AUROC por entidad federativa

Aquí aparece el motivo de la tesis: si el AUROC varía mucho entre entidades, el modelo es menos confiable en unas que en otras. Esta tabla es el punto de partida del notebook 03.


In [8]:
# Se reutilizan las funciones de tesis_common: el notebook 03 mide exactamente
# lo mismo con el mismo código, así que sus cifras y las de aquí son comparables.
y_te_arr = y_te.to_numpy()
g_te_arr = g_te.to_numpy()

tab = tc.metricas_por_grupo(y_te_arr, p_hgb, g_te_arr, UMBRAL).sort_values("AUROC")
_, descartadas = tc.grupos_evaluables(g_te_arr, y_te_arr)

print(tab[["grupo", "n", "tasa", "AUROC", "TPR", "FPR"]].round(4).to_string(index=False))
print("\nAUROC global:", round(roc_auc_score(y_te, p_hgb), 4))
print("Rango entre entidades:", round(tab["AUROC"].min(), 3), "a", round(tab["AUROC"].max(), 3),
      "| brecha:", round(tab["AUROC"].max() - tab["AUROC"].min(), 3))

# Reportar lo descartado evita que una cobertura parcial se lea como completa.
if len(descartadas):
    print("\nEntidades sin soporte suficiente (excluidas):")
    print(descartadas.to_string(index=False))
else:
    print("\nTodas las entidades tienen soporte suficiente "
          f"(n>={tc.MIN_N_GRUPO}, positivos>={tc.MIN_POS_GRUPO}).")

              grupo      n   tasa  AUROC    TPR    FPR
          CHIHUAHUA  11072 0.0713 0.9207 0.6882 0.0809
            CHIAPAS   3831 0.0611 0.9265 0.6624 0.0648
BAJA CALIFORNIA SUR  11712 0.0307 0.9272 0.6833 0.0396
     AGUASCALIENTES   5498 0.0615 0.9304 0.6568 0.0601
            DURANGO   6297 0.0569 0.9310 0.6313 0.0520
           VERACRUZ  20275 0.1012 0.9340 0.7665 0.0855
         NUEVO LEON  32110 0.0528 0.9341 0.6865 0.0463
           COAHUILA  12615 0.0576 0.9342 0.6860 0.0537
            JALISCO  27918 0.0887 0.9348 0.7026 0.0666
          MICHOACAN   9484 0.1281 0.9359 0.8527 0.1159
             MEXICO  65705 0.0720 0.9387 0.7597 0.0696
           TLAXCALA   4288 0.0702 0.9391 0.8405 0.1003
            SINALOA  12274 0.0849 0.9394 0.7994 0.0756
            NAYARIT   6700 0.0710 0.9402 0.6996 0.0493
            HIDALGO   9345 0.1022 0.9410 0.8806 0.1226
          ZACATECAS   5534 0.0614 0.9431 0.7735 0.0674
            YUCATAN  12490 0.0705 0.9440 0.7261 0.0475
          

## 9. Calibración y guardado

In [9]:
import json

# Calibración del modelo que efectivamente se despliega y audita (calibrado).
frac_pos, mean_pred = calibration_curve(y_te, p_hgb, n_bins=10, strategy="quantile")
print("Calibración (deciles de riesgo predicho):")
for mp, fp in zip(mean_pred, frac_pos):
    print(f"  pred~{mp:.3f} -> observado {fp:.3f}")

# Se guarda un PAQUETE (modelo + calibrador + huella de cohorte y predictores),
# no el estimador desnudo: así los notebooks 03 y 04 no pueden auditar un modelo
# entrenado sobre otra cohorte ni olvidar aplicar la calibración. Antes cada uno
# reentrenaba su propio modelo —y el 04 con otros hiperparámetros—.
ruta = tc.guardar_modelo(hgb, calibrador, n_filas_cohorte=len(df))
print("\nPaquete del modelo guardado:", ruta)
print("Hiperparámetros canónicos:", tc.HPARAMS_HGB)

# Exportación de las cifras de desempeño que cita el documento, para que la prosa
# no las transcriba a mano.
modelo_ref = {
    "n_train": int(len(X_tr)), "n_calib": int(len(X_cal)), "n_test": int(len(X_te)),
    "prevalencia": round(float(y.mean()), 6),
    "desbalance": round(float((1 - y.mean()) / y.mean()), 2),
    "logit": {
        "auroc": round(float(roc_auc_score(y_te, p_logit)), 4),
        "auprc": round(float(average_precision_score(y_te, p_logit)), 4),
        "brier": round(float(brier_score_loss(y_te, p_logit)), 4),
    },
    "hgb_crudo": {
        "auroc": round(float(roc_auc_score(y_te, p_te_crudo)), 4),
        "auprc": round(float(average_precision_score(y_te, p_te_crudo)), 4),
        "brier": round(float(brier_score_loss(y_te, p_te_crudo)), 4),
        "riesgo_medio": round(float(p_te_crudo.mean()), 4),
    },
    "hgb_calibrado": {
        "auroc": round(float(roc_auc_score(y_te, p_hgb)), 4),
        "auprc": round(float(average_precision_score(y_te, p_hgb)), 4),
        "brier": round(float(brier_score_loss(y_te, p_hgb)), 4),
        "riesgo_medio": round(float(p_hgb.mean()), 4),
    },
    "mortalidad_observada_test": round(float(y_te.mean()), 4),
    "umbral": round(float(UMBRAL), 4),
    "tpr_objetivo": round(float(tpr_obj), 4),
    "fpr_objetivo": round(float(fpr_obj), 4),
    "importancia_permutacion": [
        {"variable": k, "importancia": round(float(v), 4)}
        for k, v in imp_df.items()
    ],
    "calibracion_deciles": [
        {"predicho": round(float(mp), 4), "observado": round(float(fp), 4)}
        for mp, fp in zip(mean_pred, frac_pos)
    ],
}
with open("modelo_referencia.json", "w", encoding="utf-8") as fh:
    json.dump(modelo_ref, fh, indent=2, ensure_ascii=False)
print("Desempeño exportado: modelo_referencia.json")

Calibración (deciles de riesgo predicho):
  pred~0.000 -> observado 0.000
  pred~0.001 -> observado 0.001
  pred~0.001 -> observado 0.001
  pred~0.002 -> observado 0.002
  pred~0.003 -> observado 0.003
  pred~0.006 -> observado 0.006
  pred~0.011 -> observado 0.011
  pred~0.024 -> observado 0.023
  pred~0.077 -> observado 0.076
  pred~0.444 -> observado 0.447

Paquete del modelo guardado: modelo_hgb_mortalidad.joblib
Hiperparámetros canónicos: {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 6, 'class_weight': 'balanced', 'random_state': 42}


Desempeño exportado: modelo_referencia.json


## 10. Análisis de sensibilidad — subconjunto hospitalizado

En la cohorte de confirmados, **fallecer implica estar registrado como hospitalizado**: no hay ninguna defunción ambulatoria (notebook 01). El desenlace solo es posible, por tanto, en el 11.8 % hospitalizado de la cohorte, y el 88 % restante son casos que no pueden fallecer por construcción.

Eso no invalida la tarea —predecir riesgo al primer contacto sobre todo caso confirmado es exactamente el triage que motiva la tesis— pero sí condiciona cómo debe leerse el AUROC: una parte de la discriminación consiste en separar ambulatorios de hospitalizados, algo que `NEUMONIA` al primer contacto hace muy bien.

Esta sección reentrena el mismo modelo restringido a los pacientes hospitalizados, es decir, a la población donde la muerte es posible. Es la cifra comparable con los modelos clínicos de mortalidad hospitalaria y con cualquier validación externa sobre bases de egresos, y sirve además como prueba de robustez del diagnóstico de equidad.

In [10]:
# Mismo modelo, mismos hiperparámetros y misma partición: la única diferencia es
# la población. Se reentrena porque restringir a posteriori el conjunto de prueba
# del modelo global mediría otra cosa (el modelo habría visto ambulatorios).
sub = df[df["hospitalizado"] == 1].copy()
Xh = sub[PREDICTORES].astype("float32")
yh = sub[TARGET].astype(int)
gh = sub[SENSIBLE]

ih_tr, ih_cal, ih_te = tc.particionar(yh)
hgb_h = HistGradientBoostingClassifier(**tc.HPARAMS_HGB).fit(Xh.iloc[ih_tr], yh.iloc[ih_tr])
cal_h = IsotonicRegression(out_of_bounds="clip").fit(
    hgb_h.predict_proba(Xh.iloc[ih_cal])[:, 1], yh.iloc[ih_cal])
p_h = cal_h.predict(hgb_h.predict_proba(Xh.iloc[ih_te])[:, 1])

yh_te = yh.iloc[ih_te].to_numpy()
gh_te = gh.iloc[ih_te].to_numpy()
UMBRAL_H, tpr_h, fpr_h = tc.umbral_para_sensibilidad(yh_te, p_h)
tab_h = tc.metricas_por_grupo(yh_te, p_h, gh_te, UMBRAL_H)
dis_h = tc.resumen_disparidad(tab_h)

# El AUPRC hay que leerlo contra la prevalencia: su linea base ES la prevalencia,
# que aqui pasa de 5.6% a 47.5%. Sin normalizar, la comparacion enganaria.
prev_g, prev_h = float(y.mean()), float(yh.mean())
auprc_g = average_precision_score(y_te, p_hgb)
auprc_h = average_precision_score(yh_te, p_h)

print(f"Cohorte completa : n={len(df):,}  prevalencia={prev_g:.3f}")
print(f"  AUROC {roc_auc_score(y_te, p_hgb):.4f} | AUPRC {auprc_g:.4f} "
      f"({auprc_g/prev_g:.2f}x la prevalencia) | Brier {brier_score_loss(y_te, p_hgb):.4f}")
print(f"  gap_TPR {tab['TPR'].max()-tab['TPR'].min():.4f} | "
      f"gap_AUROC {tab['AUROC'].max()-tab['AUROC'].min():.4f}")
print(f"\nSolo hospitalizados: n={len(sub):,}  prevalencia={prev_h:.3f}")
print(f"  AUROC {roc_auc_score(yh_te, p_h):.4f} | AUPRC {auprc_h:.4f} "
      f"({auprc_h/prev_h:.2f}x la prevalencia) | Brier {brier_score_loss(yh_te, p_h):.4f}")
print(f"  gap_TPR {dis_h['gap_TPR (equalized odds)']:.4f} | "
      f"gap_AUROC {dis_h['gap_AUROC']:.4f}")
t_h = tab_h.sort_values("AUROC")
print(f"  peor {t_h.iloc[0]['grupo']} {t_h.iloc[0]['AUROC']:.4f} | "
      f"mejor {t_h.iloc[-1]['grupo']} {t_h.iloc[-1]['AUROC']:.4f}")
print("\n -> en la poblacion en riesgo la brecha se desplaza del umbral al ordenamiento.")

# --- Segunda prueba: ¿depende el resultado de NEUMONIA? -----------------------
# NEUMONIA domina la importancia por permutación y su captura al primer contacto
# es un supuesto del que penden los resultados. Se reentrena sin ella para acotar
# cuánto del desempeño y de la inequidad descansan en esa única variable.
PRED_SIN_NEU = [c for c in PREDICTORES if c != "NEUMONIA_bin"]
Xn = df[PRED_SIN_NEU].astype("float32")
hgb_n = HistGradientBoostingClassifier(**tc.HPARAMS_HGB).fit(Xn.iloc[idx_tr], y_tr)
cal_n = IsotonicRegression(out_of_bounds="clip").fit(
    hgb_n.predict_proba(Xn.iloc[idx_cal])[:, 1], y_cal)
p_n = cal_n.predict(hgb_n.predict_proba(Xn.iloc[idx_te])[:, 1])

UMBRAL_N, tpr_n, _ = tc.umbral_para_sensibilidad(y_te_arr, p_n)
tab_n = tc.metricas_por_grupo(y_te_arr, p_n, g_te_arr, UMBRAL_N)
dis_n = tc.resumen_disparidad(tab_n)

print(f"\nSin NEUMONIA ({len(PRED_SIN_NEU)} predictores):")
print(f"  AUROC {roc_auc_score(y_te, p_n):.4f} (con NEUMONIA {roc_auc_score(y_te, p_hgb):.4f})"
      f" | AUPRC {average_precision_score(y_te, p_n):.4f}")
print(f"  gap_TPR {dis_n['gap_TPR (equalized odds)']:.4f} "
      f"(con NEUMONIA {tab['TPR'].max()-tab['TPR'].min():.4f})")
print(f"  gap_AUROC {dis_n['gap_AUROC']:.4f} "
      f"(con NEUMONIA {tab['AUROC'].max()-tab['AUROC'].min():.4f})")
print(" -> si la brecha persiste sin NEUMONIA, la inequidad no es un artefacto de")
print("    la practica de registro de esa variable.")

sensibilidad_neumonia = {
    "n_predictores": len(PRED_SIN_NEU),
    "auroc": round(float(roc_auc_score(y_te, p_n)), 4),
    "auprc": round(float(average_precision_score(y_te, p_n)), 4),
    "brier": round(float(brier_score_loss(y_te, p_n)), 4),
    "umbral": round(float(UMBRAL_N), 4),
    "gap_TPR": round(float(dis_n["gap_TPR (equalized odds)"]), 4),
    "gap_AUROC": round(float(dis_n["gap_AUROC"]), 4),
    "caida_auroc": round(float(roc_auc_score(y_te, p_hgb) - roc_auc_score(y_te, p_n)), 4),
}

sensibilidad = {
    "completa": {"n": int(len(df)), "prevalencia": round(prev_g, 4),
                 "auroc": round(float(roc_auc_score(y_te, p_hgb)), 4),
                 "auprc": round(float(auprc_g), 4),
                 "auprc_sobre_prevalencia": round(float(auprc_g / prev_g), 2),
                 "brier": round(float(brier_score_loss(y_te, p_hgb)), 4),
                 "gap_TPR": round(float(tab["TPR"].max() - tab["TPR"].min()), 4),
                 "gap_AUROC": round(float(tab["AUROC"].max() - tab["AUROC"].min()), 4)},
    "hospitalizados": {"n": int(len(sub)), "prevalencia": round(prev_h, 4),
                       "auroc": round(float(roc_auc_score(yh_te, p_h)), 4),
                       "auprc": round(float(auprc_h), 4),
                       "auprc_sobre_prevalencia": round(float(auprc_h / prev_h), 2),
                       "brier": round(float(brier_score_loss(yh_te, p_h)), 4),
                       "umbral": round(float(UMBRAL_H), 4),
                       "gap_TPR": round(float(dis_h["gap_TPR (equalized odds)"]), 4),
                       "gap_AUROC": round(float(dis_h["gap_AUROC"]), 4),
                       "peor_entidad": t_h.iloc[0]["grupo"],
                       "peor_auroc": round(float(t_h.iloc[0]["AUROC"]), 4),
                       "mejor_entidad": t_h.iloc[-1]["grupo"],
                       "mejor_auroc": round(float(t_h.iloc[-1]["AUROC"]), 4),
                       "por_entidad": tab_h.round(4).to_dict(orient="records")},
    "sin_neumonia": sensibilidad_neumonia,
}
with open("sensibilidad_hospitalizados.json", "w", encoding="utf-8") as fh:
    json.dump(sensibilidad, fh, indent=2, ensure_ascii=False)
print("\nExportado: sensibilidad_hospitalizados.json")

Cohorte completa : n=2,526,649  prevalencia=0.056
  AUROC 0.9503 | AUPRC 0.5533 (9.88x la prevalencia) | Brier 0.0323
  gap_TPR 0.2935 | gap_AUROC 0.0563

Solo hospitalizados: n=297,685  prevalencia=0.475
  AUROC 0.7001 | AUPRC 0.6370 (1.34x la prevalencia) | Brier 0.2179
  gap_TPR 0.1876 | gap_AUROC 0.1494
  peor CHIAPAS 0.6110 | mejor TABASCO 0.7604

 -> en la poblacion en riesgo la brecha se desplaza del umbral al ordenamiento.



Sin NEUMONIA (12 predictores):
  AUROC 0.8893 (con NEUMONIA 0.9503) | AUPRC 0.3311
  gap_TPR 0.1529 (con NEUMONIA 0.2935)
  gap_AUROC 0.0642 (con NEUMONIA 0.0563)
 -> si la brecha persiste sin NEUMONIA, la inequidad no es un artefacto de
    la practica de registro de esa variable.



Exportado: sensibilidad_hospitalizados.json


## 10. Próximos pasos (notebook 03 — auditoría de equidad)

1. Formalizar métricas de equidad por entidad, sexo y grupo etario: diferencias en TPR/FPR (equalized odds), calibración por grupo.
2. Correlacionar la brecha de AUROC por entidad con indicadores de infraestructura/marginación.
3. Explorar subgrupos interseccionales (edad × entidad).
4. Preparar la mitigación (reponderación, umbrales por grupo) y la frontera de Pareto — notebook 04.

> Recordatorio: la entidad se usa **solo** como atributo sensible para auditar, no como predictor del modelo desplegado.
